In [ ]:
# Fabric notebook source

# METADATA ********************

# META {
# META   "kernel_info": {
# META     "name": "synapse_pyspark"
# META   },
# META   "dependencies": {
# META     "lakehouse": {
# META       "default_lakehouse": "1a2dac56-0173-4466-bf42-2b813eb64114",
# META       "default_lakehouse_name": "Lakehouse_RavenDb_Idia",
# META       "default_lakehouse_workspace_id": "41d8931a-670b-4f64-a839-c7495b39aa59",
# META       "known_lakehouses": [
# META         {
# META           "id": "1a2dac56-0173-4466-bf42-2b813eb64114"
# META         }
# META       ]
# META     }
# META   }
# META }

# MARKDOWN ********************

# # Silver Customers — Landing → Silver Upsert
# 
# Reads Parquet files from the landing zone, applies a **watermark** to skip
# already-processed files, and merges new rows into the `dbo.silver_customers`
# Delta table using a configurable upsert key.

# CELL ********************

# ============================================================
# 1. IMPORTS & CONFIGURATION
# ============================================================

import re
from datetime import datetime

import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from delta.tables import DeltaTable

# ── Lakehouse paths ──────────────────────────────────────────
LANDING_PATH      = "abfss://.../.../Landing/customers"
SILVER_TABLE_PATH = "abfss://.../.../Tables/dbo/silver_customers"
SILVER_TABLE_NAME = "dbo.silver_customers"

# ── Upsert key (column that uniquely identifies a record) ────
UPSERT_KEY = "CustomerId"

# ── Filename timestamp pattern: YYYY-MM-DD-HH-MM-SS.micros ──
_TS_PATTERN = re.compile(r"(\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}\.\d+)")
_TS_FORMAT  = "%Y-%m-%d-%H-%M-%S.%f"

# Allow Delta MERGE to add new columns from source automatically
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# METADATA ********************

# META {
# META   "language": "python",
# META   "language_group": "synapse_pyspark"
# META }

# CELL ********************

# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def extract_timestamp_from_filename(path: str) -> str | None:
    """Return the raw timestamp string embedded in a filename, or None."""
    match = _TS_PATTERN.search(path)
    return match.group(1) if match else None


def parse_file_timestamp(path: str) -> datetime | None:
    """Parse the timestamp embedded in *path* into a datetime, or None."""
    ts_str = extract_timestamp_from_filename(path)
    return datetime.strptime(ts_str, _TS_FORMAT) if ts_str else None


def list_parquet_files(spark, path: str) -> list[str]:
    """Return the full ABFSS paths of every .parquet file under *path*."""
    jvm   = spark._jvm
    fs    = jvm.org.apache.hadoop.fs.FileSystem.get(
                jvm.org.apache.hadoop.fs.Path(path).toUri(),
                spark._jsc.hadoopConfiguration()
            )
    return [
        str(s.getPath())
        for s in fs.listStatus(jvm.org.apache.hadoop.fs.Path(path))
        if str(s.getPath()).endswith(".parquet")
    ]


def quote(col_name: str) -> str:
    """Backtick-escape a column name for Delta SET / MERGE expressions."""
    return f"`{col_name}`"

# METADATA ********************

# META {
# META   "language": "python",
# META   "language_group": "synapse_pyspark"
# META }

# CELL ********************

# ============================================================
# 3. WATERMARK — LIST FILES & FILTER ALREADY-PROCESSED ONES
# ============================================================

all_files = list_parquet_files(spark, LANDING_PATH)

# Load the highest file_timestamp already written to the silver table
if spark.catalog.tableExists(SILVER_TABLE_NAME):
    last_ts = spark.read.table(SILVER_TABLE_NAME).agg(F.max("file_timestamp")).collect()[0][0]
    last_ts = last_ts if isinstance(last_ts, datetime) else last_ts.to_pydatetime() if last_ts else None
else:
    last_ts = None

print(f"Last processed timestamp : {last_ts}")
print(f"Total files in landing   : {len(all_files)}")

# Sort chronologically and drop files at or before the watermark
files_with_ts = sorted(
    [(p, ts) for p, ts in ((p, parse_file_timestamp(p)) for p in all_files) if ts],
    key=lambda x: x[1],
)
files_to_process = [(p, ts) for p, ts in files_with_ts if not last_ts or ts > last_ts]

print(f"Files to process         : {len(files_to_process)}")

# METADATA ********************

# META {
# META   "language": "python",
# META   "language_group": "synapse_pyspark"
# META }

# CELL ********************

# ============================================================
# 4. ENSURE SILVER TABLE EXISTS
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE_NAME} (
        {UPSERT_KEY} STRING
    )
    USING DELTA
""")

# METADATA ********************

# META {
# META   "language": "python",
# META   "language_group": "synapse_pyspark"
# META }

# CELL ********************

# ============================================================
# 5. PROCESS FILES — UPSERT INTO SILVER (Delta MERGE)
# ============================================================

extract_ts_udf   = F.udf(extract_timestamp_from_filename, StringType())
merge_condition  = f"CAST(target.{UPSERT_KEY} AS STRING) = CAST(source.{UPSERT_KEY} AS STRING)"
total_upserted   = 0

for file_path, file_ts in files_to_process:
    print(f"\n→ {file_path.split('/')[-1]}")

    df_file = (
        spark.read.format("parquet")
        .load(file_path)
        .withColumn("_source_file",    F.lit(file_path))
        .withColumn("file_timestamp",  F.to_timestamp(F.lit(str(file_ts)), "yyyy-MM-dd HH:mm:ss.SSSSSS"))
    )

    row_count   = df_file.count()
    update_cols = {quote(c): f"source.{quote(c)}" for c in df_file.columns}

    if DeltaTable.isDeltaTable(spark, SILVER_TABLE_PATH):
        (
            DeltaTable.forPath(spark, SILVER_TABLE_PATH).alias("target")
            .merge(df_file.alias("source"), merge_condition)
            .whenMatchedUpdate(set=update_cols)
            .whenNotMatchedInsert(values=update_cols)
            .execute()
        )
    else:
        # First file: bootstrap the Delta table
        (
            df_file.write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
            .save(SILVER_TABLE_PATH)
        )

    total_upserted += row_count
    print(f"   ✓ {row_count:,} rows upserted")

spark.catalog.refreshTable(SILVER_TABLE_NAME)

# METADATA ********************

# META {
# META   "language": "python",
# META   "language_group": "synapse_pyspark"
# META }

# CELL ********************

# ============================================================
# 6. SUMMARY
# ============================================================

if files_to_process:
    print(f"Done. {total_upserted:,} rows upserted across {len(files_to_process)} file(s).")
else:
    print("No new files — silver table is already up to date.")

total_records = spark.read.table(SILVER_TABLE_NAME).count()
print(f"Total records in silver table: {total_records:,}")

# METADATA ********************

# META {
# META   "language": "python",
# META   "language_group": "synapse_pyspark"
# META }

# CELL ********************

# ============================================================
# 7. SPOT-CHECK — PREVIEW SILVER TABLE
# ============================================================

df_silver = spark.sql(f"SELECT * FROM {SILVER_TABLE_NAME} LIMIT 1000")
display(df_silver)

# METADATA ********************

# META {
# META   "language": "python",
# META   "language_group": "synapse_pyspark"
# META }


StatementMeta(, fadb64df-1e1a-42ef-9f85-a659515d1502, 3, Finished, Available, Finished, False)

Last processed timestamp : 2026-03-23 17:22:00.001105
Total files in landing   : 5
Files to process         : 3

→ 2026-03-23-17-37-00.033156-ravendbpoc-ERP-to-ADLS-DELETION-CustomersTrans.parquet
   ✓ 1 rows upserted

→ 2026-03-23-17-55-03.786412-ravendbpoc-ERP-to-ADLS-DELETION-SimpleTransformations.parquet
   ✓ 20 rows upserted

→ 2026-03-23-17-56-00.008148-ravendbpoc-ERP-to-ADLS-DELETION-SimpleTransformations.parquet
   ✓ 1 rows upserted
Done. 22 rows upserted across 3 file(s).
Total records in silver table: 22


SynapseWidget(Synapse.DataFrame, 78c9e78d-a55d-48eb-9d8a-220fe275db22)

In [2]:
df = spark.sql("SELECT * FROM Lakehouse_RavenDb_Idia.dbo.silver_customers LIMIT 1000")
display(df)

StatementMeta(, fadb64df-1e1a-42ef-9f85-a659515d1502, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, de60c634-a426-4011-b41d-dee209584013)

In [3]:
#spark.sql("Delete from Lakehouse_RavenDb_Idia.dbo.silver_customers")


StatementMeta(, fadb64df-1e1a-42ef-9f85-a659515d1502, 5, Finished, Available, Finished, False)